# 9 · BM25

**Owner:** Member 3 · **Branch:** `dev-bm25` · **Code:** `src/discovery/rank/bm25.py` → class `BM25Ranker` · **Writes to:** `eval/results/bm25.json`

**What it means:** BM25 adds up, for each query word, how rare the word is (IDF) times a **saturating** term frequency: the 1st occurrence counts a lot, the 10th barely more. It also scales term frequency by README length, so a long README needs more occurrences for the same score.

**Key idea:** Those two corrections are why BM25 usually beats TF-IDF (hypothesis H1): `k1` controls how fast repeated words stop counting, `b` how much README length is corrected for. It is our default method.

Full guide with starter code, and the Results sheet: [team plan](https://claude.ai/artifact/Rst8GHxAbeHavg4JS2RZV3) (tab 9 BM25). Handoff and setup: `docs/handoff-methods.md`.

### Settings to try
| Setting | Values | Why |
|---|---|---|
| k1 | 0.9, 1.2, 1.5, 2.0 | Saturation: small k1 = repeating a word barely helps. |
| b | 0.3, 0.5, 0.75, 0.9 | Length correction: 0 = none, 1 = full. |

**Fixed:** idf = log((N − n + 0.5) / (n + 0.5) + 1), never negative; our own implementation, checked against rank-bm25

### What we agreed (same for all five)
- **One tokenizer:** `tokenize()` from `discovery.preprocess.text`, for documents *and* queries. Never a private one.
- **One interface:** your class extends `SearchMethod` (`src/discovery/rank/base.py`): `fit(corpus)` → self, `search(query, k)` → `[{"doc_id": full_name, "score": float}]`, best first. `check_hits()` checks it.
- **Tune on the 5 development queries only** (`DEV`, replaced by the real ones in Phase 13). The 20 test queries are for the final number only.
- **Seed 42**, paths from `PATHS`, and don't edit the shared files (`rank/base.py`, `rank/devkit.py`, `preprocess/text.py`, `config/config.yaml`) on your branch: ask Member 6.
- Commit this notebook **with its outputs** on `dev-bm25`. No data files in git.

## 1. Setup: load the corpus and the shared tokenizer

**Check:** it prints 8,223 repositories, and `node.js` and `c#` come out of `tokenize()` in one piece.

In [ ]:
%load_ext autoreload
%autoreload 2

import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from discovery.config import PATHS, SEED, set_seed
from discovery.preprocess.text import doc_text, tokenize      # the ONE shared tokenizer (Phase 6)
from discovery.rank.base import check_hits
from discovery.rank.devkit import DEV, load_corpus, measure, save_results, show_top
from discovery.rank.bm25 import BM25Ranker

set_seed()
corpus = load_corpus()                                   # data/processed/corpus.parquet
ids = corpus["full_name"].tolist()                       # doc_id = full_name, "owner/name"
text = doc_text(corpus).tolist()                         # name + description + topics + README
print(f"{len(corpus):,} repositories |", tokenize("Built with Node.js and C#"))
print("development queries:", *DEV, sep="\n  ")

## 2. Build: BM25

Everything BM25 needs is counted once: document lengths, the average length, document frequency per word, and posting lists with term frequencies.

**Check:** it runs once on the whole corpus without errors. Write down how long it took: it goes in the sheet.

<details><summary>Starter code from the team plan (open it if you are stuck; adapt it, don't just paste)</summary>

```python
import numpy as np
from collections import Counter, defaultdict

docs = [tokenize(t) for t in text]
N = len(docs)
dl = np.array([len(d) for d in docs]); avgdl = dl.mean()
tfs = [Counter(d) for d in docs]
df = Counter(w for c in tfs for w in c)
idf = {w: np.log((N - n + 0.5) / (n + 0.5) + 1) for w, n in df.items()}
inv = defaultdict(list)                              # term -> [(row, tf)]: a query only touches its own words
for i, c in enumerate(tfs):
    for w, f in c.items():
        inv[w].append((i, f))
print(f"N = {N:,} | average length {avgdl:.0f} tokens | idf('nestjs') = {idf.get('nestjs', 0):.2f}, idf('app') = {idf.get('app', 0):.2f}")
```
</details>

## 3. Search: a free-text query in, a ranked list out

The formula, term by term. Only repositories that contain at least one query word get a score.

**Check:** the top 10 for `DEV[0]` look plausible when you read their names and descriptions.

<details><summary>Starter code from the team plan (open it if you are stuck; adapt it, don't just paste)</summary>

```python
def search(query, k=10, k1=1.5, b=0.75):
    s = np.zeros(N)
    for w in tokenize(query):
        for i, f in inv.get(w, ()):
            s[i] += idf[w] * f * (k1 + 1) / (f + k1 * (1 - b + b * dl[i] / avgdl))
    top = np.argsort(-s)[:k]
    return [{"doc_id": ids[i], "score": float(s[i])} for i in top if s[i] > 0]

search(DEV[0])
```
</details>

## 4. Move it into `src/discovery/rank/bm25.py` as `BM25Ranker`

The stub is already there with the right signature. Fill in `fit` and `search` (and `params`), then load it here with `discovery.rank.bm25.BM25Ranker`: autoreload picks up your edits.

**Check:** `check_hits` passes below, and in a terminal `pytest tests/test_methods.py -k bm25` passes (it was skipped until now).

<details><summary>The search method, as in the plan (open it if you are stuck; adapt it, don't just paste)</summary>

```python
    def search(self, query, k=10):
        ...  # the function above, with self.k1, self.b
```
</details>

In [ ]:
method = BM25Ranker().fit(corpus)
check_hits(method.search(DEV[0], 10), k=10)
method.params()

## 5. Compare against a reference BM25 (rank-bm25)

`rank_bm25.BM25Okapi` uses a slightly different idf (it can go negative for very common words and floors them), so scores differ a little; the top 10 should mostly agree.

**Check:** 8+ of 10 the same on each development query, and your hand calculation for one repository matches the code.

<details><summary>The check, as in the plan (open it if you are stuck; adapt it, don't just paste)</summary>

```python
from rank_bm25 import BM25Okapi

ref = BM25Okapi(docs, k1=1.5, b=0.75)
for q in DEV:
    ours = [h["doc_id"] for h in search(q)]
    r = ref.get_scores(tokenize(q))
    theirs = [ids[i] for i in np.argsort(-r)[:10]]
    print(f"{len(set(ours) & set(theirs)):2d}/10 the same | {q}")

# one repository by hand: the score from the formula on paper = the score from the code
i = ids.index(search(DEV[0])[0]["doc_id"])
print({w: (tfs[i][w], round(idf.get(w, 0), 2)) for w in tokenize(DEV[0])}, "length", dl[i])
```
</details>

## 6. Read your top 10 on the development queries

Read them. Write down two good results and two bad ones, and why each bad one ranked high (in the markdown cell under the output).

**Check:** you can say in one sentence why each bad result ranked high.

In [ ]:
show_top(method, corpus, DEV)

*Two good results, two bad results, and why:*

- 
- 
- 
- 

## 7. Tune on the 5 development queries only

Grid over k1 × b (16 combinations) on the 5 development queries, mean NDCG@10. Plot it as a heatmap: it is report material.

Until Phase 13 gives the development labels, compare settings by reading the top 10 (step 6). When `eval/qrels.csv` has the development labels, use `discovery.evaluation.metrics.ndcg_at_k` and keep the setting with the best mean NDCG@10. **Never tune on the 20 test queries.**

**Check:** a small table: each setting tried, what changed in the top 10 (later: mean NDCG@10). Keep the best and note it in the sheet.

## 8. Speed, results file, hand in

`measure()` times the build and the queries; `save_results()` writes `eval/results/bm25.json` in the format the Results sheet imports. The metrics stay empty until Phase 13–14: then Member 6's `evaluate_method(method, split="test")` fills them in and you import the file in the **Results sheet** tab of the team plan, add your observations and download the PDF.

**Check:** `eval/results/bm25.json` exists; this notebook is committed with its outputs on `dev-bm25`; `pytest` passes.

In [ ]:
speed = measure(lambda: BM25Ranker().fit(corpus), DEV)
print(speed)
save_results(method, speed)

## Extra (optional): BM25F: score the fields separately

Give name and topics their own length normalisation and a weight (for example name × 3), then add the fields up. Compare with plain BM25 on the development queries.

## What we found

*(fill in at the end: build time, the best setting, two good and two bad results, and anything surprising)*